<a href="https://colab.research.google.com/github/yashb98/90Days_Machine_learinng/blob/main/Synapse.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Install Dependencies



In [ ]:

!pip install -q streamlit # -q for "quiet"
!pip install -q langchain langchain-openai llama-index openai faiss-cpu sentence-transformers pandas python-dotenv
!pip install -q openai-whisper

# Install ffmpeg for audio processing
!apt-get install -y -qq ffmpeg

In [ ]:
!pip install datasets

## Create Directories & Download MTS-Dialog

In [ ]:

import os
from datasets import load_dataset

print("Creating directory structure...")
os.makedirs("data/ehr", exist_ok=True)
os.makedirs("data/golden_path", exist_ok=True)
!ls -R data

print("\nDownloading MTS-Dialog dataset from Hugging Face...")
# This dataset has 'train', 'validation', 'test' splits. We'll use 'train'.
try:
    mts_dataset = load_dataset("har1/MTS_Dialogue-Clinical_Note", split='train')
    print("\nMTS-Dialog dataset loaded successfully.")
    print(f"Total samples: {len(mts_dataset)}")

    # Let's inspect the first sample
    print("\n--- Sample 1 ---")
    print(f"[DIALOGUE]:\n{mts_dataset[0]['dialogue']}")
    print(f"\n[NOTE]:\n{mts_dataset[0]['note']}")
    print("------------------")

except Exception as e:
    print(f"Error loading dataset: {e}")


## Mount Google Drive

In [ ]:

from google.colab import drive
import os

print("Mounting Google Drive...")
drive.mount('/content/drive')

# --- VERIFY YOUR PATH ---
# This command lists the files in your 'csv' folder.
# If this command fails, your folder isn't at 'My Drive/csv'.
# Adjust the path as needed.
print("\nVerifying access to your Synthea files...")
!ls -lh /content/drive/MyDrive/csv

### Full Column Name Diagnostic

In [ ]:

import pandas as pd
from pathlib import Path
import os

# Path to your Synthea CSVs in Google Drive
CSV_DIR = Path("/content/drive/MyDrive/csv")

# List of all 18 CSV files from your screenshot
csv_files_list = [
    "allergies.csv",
    "careplans.csv",
    "claims_transactions.csv",
    "claims.csv",
    "conditions.csv",
    "devices.csv",
    "encounters.csv",
    "imaging_studies.csv",
    "immunizations.csv",
    "medications.csv",
    "observations.csv",
    "organizations.csv",
    "patients.csv",
    "payer_transitions.csv",
    "payers.csv",
    "procedures.csv",
    "providers.csv",
    "supplies.csv"
]

print(f"--- Reading all column headers from {CSV_DIR} ---")
print("This will check all 18 files...\n")

missing_files = []

# Loop through each file
for file_name in csv_files_list:
    file_path = CSV_DIR / file_name

    # Check if file exists
    if not file_path.exists():
        print(f"!!! WARNING: File not found: {file_name} !!!\n")
        missing_files.append(file_name)
        continue

    # Read only the header row (nrows=0) to get columns
    try:
        df_header = pd.read_csv(file_path, nrows=0)

        print(f"--- Columns in {file_name} ---")
        print(df_header.columns.tolist())
        print("--------------------------------" + "-" * len(file_name) + "\n")

    except pd.errors.EmptyDataError:
        print(f"--- {file_name} is empty ---")
        print("[]")
        print("-----------------------" + "-" * len(file_name) + "\n")
    except Exception as e:
        print(f"!!! Error reading {file_name}: {e} !!!\n")

if missing_files:
    print(f"\nSummary: Could not find the following files: {missing_files}")
else:
    print("\nSummary: All 18 files were found and headers were read successfully.")

## Define the ENHANCED Synthea processing script

In [ ]:

import pandas as pd
from pathlib import Path
import os

# Path to your Synthea CSVs in Google Drive
CSV_DIR = Path("/content/drive/MyDrive/csv")
OUTPUT_DIR = Path("data/ehr")

def process_synthea_data_enhanced():
    """
    Reads multiple Synthea CSVs from Google Drive and creates one rich
    .txt file per patient in the Colab 'data/ehr/' directory.

    (Version 2 - Corrected DATE/START key error)
    """
    print(f"Reading CSVs from: {CSV_DIR}")

    if not CSV_DIR.exists():
        print(f"Error: Directory not found: {CSV_DIR}")
        return

    try:
        patients = pd.read_csv(CSV_DIR / "patients.csv")
        meds = pd.read_csv(CSV_DIR / "medications.csv")
        conditions = pd.read_csv(CSV_DIR / "conditions.csv")
        allergies = pd.read_csv(CSV_DIR / "allergies.csv")
        procedures = pd.read_csv(CSV_DIR / "procedures.csv")
        encounters = pd.read_csv(CSV_DIR / "encounters.csv")
        observations = pd.read_csv(CSV_DIR / "observations.csv")
    except FileNotFoundError as e:
        print(f"Error loading file: {e}")
        print(f"Please ensure all CSV files (patients, meds, conditions, etc.) are in your folder: {CSV_DIR}")
        return
    except Exception as e:
        print(f"An error occurred: {e}")
        return

    # Create output directory
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    print(f"Processing {len(patients)} patients...")

    # Process each patient
    for _, patient in patients.iterrows():
        patient_id = patient["Id"]

        # 1. Demographics
        patient_info = [
            f"Patient ID: {patient_id}",
            f"Name: {patient['FIRST']} {patient['LAST']}",
            f"Gender: {patient['GENDER']}",
            f"Birthdate: {patient['BIRTHDATE']}",
            f"Address: {patient.get('ADDRESS', 'N/A')}",
            f"Marital Status: {patient.get('MARITAL', 'N/A')}",
        ]

        # 2. Allergies
        patient_allergies = allergies[allergies["PATIENT"] == patient_id]
        allergy_list = [f"- {desc}" for desc in patient_allergies["DESCRIPTION"].unique()]

        # 3. Active Conditions
        patient_conditions = conditions[conditions["PATIENT"] == patient_id]
        condition_list = [f"- {desc}" for desc in patient_conditions["DESCRIPTION"].unique()]

        # 4. Current Medications
        patient_meds = meds[meds["PATIENT"] == patient_id]
        med_list = [f"- {desc}" for desc in patient_meds["DESCRIPTION"].unique()]

        # 5. Past Procedures
        patient_procs = procedures[procedures["PATIENT"] == patient_id]
        # FIX: Changed 'DATE' to 'START'
        proc_list = [f"- {row['DESCRIPTION']} (Date: {row['START']})" for _, row in patient_procs.iterrows()]

        # 6. Recent Encounters
        # FIX: Changed 'DATE' to 'START' for sorting
        patient_encs = encounters[encounters["PATIENT"] == patient_id].sort_values('START', ascending=False)
        # FIX: Changed 'row['DATE']' to 'row['START']'
        enc_list = [f"- {row['START']}: {row['DESCRIPTION']}" for _, row in patient_encs.head(5).iterrows()]

        # 7. Recent Observations (Vitals/Labs)
        # NO FIX NEEDED: 'DATE' column exists in observations.csv
        patient_obs = observations[observations["PATIENT"] == patient_id].sort_values('DATE', ascending=False)
        obs_list = [f"- {row['DATE']} {row['DESCRIPTION']}: {row['VALUE']} {row.get('UNITS', '')}" for _, row in patient_obs.head(10).iterrows()]

        # Assemble the text file content
        content = f"== PATIENT RECORD: {patient['FIRST']} {patient['LAST']} (ID: {patient_id}) ==\n\n"
        content += "== Demographics ==\n" + "\n".join(patient_info) + "\n\n"
        content += "== Allergies ==\n" + ("\n".join(allergy_list) if allergy_list else "None on record.") + "\n\n"
        content += "== Active Conditions / Problem List ==\n" + ("\n".join(condition_list) if condition_list else "None on record.") + "\n\n"
        content += "== Current Medications ==\n" + ("\n".join(med_list) if med_list else "None on record.") + "\n\n"
        content += "== Past Procedures ==\n" + ("\n".join(proc_list) if proc_list else "None on record.") + "\n\n"
        content += "== Recent Encounters (Last 5) ==\n" + ("\n".join(enc_list) if enc_list else "None on record.") + "\n\n"
        content += "== Recent Observations (Last 10) ==\n" + ("\n".join(obs_list) if obs_list else "None on record.") + "\n"

        # Write to file
        output_filename = OUTPUT_DIR / f"patient_{patient_id}.txt"
        with open(output_filename, "w", encoding="utf-8") as f:
            f.write(content)

    print(f"\nSuccessfully processed and saved {len(patients)} patient records to {OUTPUT_DIR}")
    print(f"Total files in {OUTPUT_DIR}: {len(list(OUTPUT_DIR.glob('*.txt')))}")

## Run the processing

In [ ]:

process_synthea_data_enhanced()

# Check the output - let's find a random file and print it
print("\n--- Sample Enhanced EHR File ---")
!ls data/ehr | head -n 1 | xargs -I {} head -n 25 data/ehr/{}

## Check MTS-Dialog Columns

In [ ]:

from datasets import load_dataset

try:
    # Ensure dataset is loaded
    mts_dataset = load_dataset("har1/MTS_Dialogue-Clinical_Note", split='train')

    # Print all column names
    print("--- Columns in MTS-Dialog Dataset ---")
    print(mts_dataset.column_names)
    print("-------------------------------------")

    # Print the first sample to see the structure
    print("\n--- First Sample Data ---")
    print(mts_dataset[0])
    print("---------------------------")

except Exception as e:
    print(f"An error occurred: {e}")

## Prepare Golden Path Data

In [ ]:

from datasets import load_dataset
import textwrap

# Load dataset (if not already loaded)
try:
    mts_dataset
except NameError:
    mts_dataset = load_dataset("har1/MTS_Dialogue-Clinical_Note", split='train')

# --- PICK YOUR GOLDEN PATH SAMPLE ---
# We'll use sample index 5. You can change this index if you want.
SAMPLE_INDEX = 5
# -----------------------------------

golden_sample = mts_dataset[SAMPLE_INDEX]
golden_dialogue = golden_sample['dialogue']

# --- THIS IS THE FIX ---
# The column is 'section_text', not 'note' or 'summary'
golden_note = golden_sample['section_text']
# ---------------------

# Save these to our golden_path folder for later
with open("data/golden_path/conversation.txt", "w", encoding="utf-8") as f:
    f.write(golden_dialogue)

with open("data/golden_path/ground_truth_note.txt", "w", encoding="utf-8") as f:
    f.write(golden_note)

print("--- PLEASE RECORD THIS DIALOGUE AS 'conversation.mp3' ---")
print(f"--- (Sample {SAMPLE_INDEX}) ---")
print("\n".join(textwrap.wrap(golden_dialogue, 80)))
print("\n---------------------------------------------------------")
print("Saved text to data/golden_path/conversation.txt")
print("Saved note to data/golden_path/ground_truth_note.txt")

## Upload 'Golden Path' Audio

In [ ]:

from google.colab import files
import os

print("Please upload your 'Golden Path' audio file (conversation.mp3)")
uploaded = files.upload()

# Move uploaded files to the correct directory
for fn in uploaded.keys():
    if fn.lower().endswith((".mp3", ".wav", ".m4a")):
        os.rename(fn, "data/golden_path/conversation.mp3")
        print(f"Renamed '{fn}' to 'conversation.mp3'")
    else:
        print(f"Warning: Uploaded file '{fn}' was not an expected audio file. Trying to rename anyway.")
        os.rename(fn, "data/golden_path/conversation.mp3")


print("\nFile uploaded to data/golden_path/:")
!ls data/golden_path

## Define and Run ASR (Whisper)

In [ ]:

import whisper
import time
from pathlib import Path

# Caching the model load
_model = None

def get_whisper_model():
    """Loads and caches the Whisper model."""
    global _model
    if _model is None:
        print("Loading Whisper model (small.en)... This may take a moment.")
        # Using "small.en" for a good balance of speed and accuracy on Colab GPU
        _model = whisper.load_model("small.en")
        print("Whisper model loaded.")
    return _model

def transcribe_audio(filepath: str) -> str:
    """Transcribes an audio file using Whisper."""
    print(f"Starting transcription for: {filepath}")
    model = get_whisper_model()

    start_time = time.time()
    try:
        # We are in Colab with a GPU, so this should be fast
        result = model.transcribe(filepath)
        end_time = time.time()
        duration = end_time - start_time
        print(f"Transcription finished in {duration:.2f} seconds.")
        return result["text"]

    except Exception as e:
        print(f"Error during transcription: {e}")
        return ""

# --- Now, let's run it ---
audio_file = "data/golden_path/conversation.mp3"
transcript_file = "data/golden_path/transcript_from_audio.txt"

if not os.path.exists(audio_file):
    print(f"Error: Audio file not found at {audio_file}")
    print("Please re-run Cell 7 to upload your file.")
else:
    # Run the transcription
    transcript = transcribe_audio(audio_file)

    if transcript:
        print("\n--- WHISPER TRANSCRIPT ---")
        print(transcript)
        print("----------------------------")

        # Save to a file for Day 35
        with open(transcript_file, "w", encoding="utf-8") as f:
            f.write(transcript)
        print(f"Transcript saved to {transcript_file}")

## Detailed Analysis and Summary of Executed Sections

Here is a detailed analysis of each executed section of the notebook, referencing them by their markdown headings, and a summary of their collective results:

*   **Install Dependencies**: This section installed crucial libraries for the project, including `streamlit` for building web applications, `langchain`, `langchain-openai`, `llama-index` for leveraging large language models and building RAG (Retrieval Augmented Generation) applications, `openai` for interacting with the OpenAI API, `faiss-cpu` for efficient similarity search (often used in RAG), `sentence-transformers` for generating embeddings, `pandas` for data manipulation, `python-dotenv` for managing environment variables (like API keys), and `openai-whisper` for Automatic Speech Recognition (ASR). It also installed `ffmpeg`, a necessary tool for handling audio files, which Whisper relies on. The `-q` flag ensured a quiet installation process. The output shows the successful installation of these packages.

*   **Create Directories**: This section used the `os.makedirs` command to create three directories: `data/synthea_csv`, `data/ehr`, and `data/golden_path`. The `exist_ok=True` argument prevents errors if the directories already exist. The `!ls -R data` command confirmed the successful creation and structure of these directories.

*   **Check MTS-Dialog Columns**: This diagnostic section specifically examined the structure of the MTS-Dialog dataset. It loaded the dataset and printed all its column names using `.column_names`. It also printed the first sample's data structure. This confirmed that the clinical note content was stored under the column name `section_text`, not `note`, resolving the `KeyError` from the initial attempt to load the dataset.

*   **Mount Google Drive**: This section used the `google.colab.drive` module to mount the user's Google Drive to the Colab environment. This is a necessary step to access the Synthea CSV files stored in the user's Drive. The `!ls -lh /content/drive/MyDrive/csv` command verified that the mounting was successful and listed the contents of the specified 'csv' folder, confirming that the Synthea files were accessible.

*   **Full Column Name Diagnostic**: This crucial diagnostic section was added to inspect the column headers of all 18 Synthea CSV files. It iterated through a predefined list of expected filenames, checked if each file existed in the mounted Google Drive directory, and used `pd.read_csv(..., nrows=0)` to efficiently read only the header row. The code then printed the list of column names for each file. This step was essential in confirming the correct column names, particularly identifying that the date/time columns were named 'START' or 'DATE' depending on the file.

*   **Define the ENHANCED Synthea processing script**: Based on the findings from the diagnostic sections, this section defined the `process_synthea_data_enhanced` function. This function reads several key Synthea CSVs (`patients`, `medications`, `conditions`, `allergies`, `procedures`, `encounters`, `observations`) and processes them to create a single text file for each patient. The function extracts demographics, allergies, conditions, medications, procedures, encounters, and observations. **Crucially, it corrected the column names used to access dates and descriptions, changing 'DATE' to 'START' for files like `procedures.csv` and `encounters.csv` where 'START' is the correct column name for the event date, and correctly using 'DATE' for `observations.csv` where that is the correct column.** It saves these structured patient records as `.txt` files in the `data/ehr` directory.

*   **Run the processing**: This section executed the `process_synthea_data_enhanced()` function defined in the previous section. It successfully processed all 1163 patients found in the `patients.csv` file and generated a corresponding text file for each in the `data/ehr` directory. The final count of files in the output directory confirmed that all patients were processed. A sample of one of the generated EHR files was printed to the output, showcasing the organized structure of the extracted patient information.

*   **Prepare Golden Path Data**: This section was updated based on the finding from the "Check MTS-Dialog Columns" section. It loaded the MTS-Dialog dataset and selected a specific sample (index 5) to serve as the "Golden Path" for testing the note generation process. It correctly extracted the dialogue from the `dialogue` column and the ground truth clinical note from the `section_text` column. These were then saved as `conversation.txt` and `ground_truth_note.txt` respectively in the `data/golden_path` directory. The dialogue was also printed for the user to record as an audio file.

*   **Upload 'Golden Path' Audio**: This section facilitated the user uploading the audio recording of the "Golden Path" dialogue. It used `google.colab.files.upload()` to open a file picker. The uploaded file (named `Conversation2.mp3` by the user) was then renamed to `conversation2.mp3` and moved into the `data/golden_path` directory. The `ls` command confirmed the file was successfully placed in the correct location alongside the text files.

*   **Define and Run ASR (Whisper)**: This section defined and executed a function `transcribe_audio` that uses the `openai-whisper` library to perform Automatic Speech Recognition on the uploaded audio file. It uses the "small.en" model, which was loaded and cached for efficiency. The function transcribed the `conversation2.mp3` file, measured the transcription time, printed the resulting text transcript, and saved it to `data/golden_path/transcript_from_audio.txt`. The output shows the successful loading of the Whisper model, the transcription process, and the final transcript.

**Overall Summary:**

The executed sections have successfully prepared the necessary components for a medical note generation application. This includes:

1.  **Environment Setup:** Installing all required libraries and utilities.
2.  **Data Preparation (Synthea):** Downloading Synthea healthcare data (though it's accessed directly from Drive in this case), processing it to create structured text records for individual patients, and saving these records in a dedicated directory (`data/ehr`). Diagnostic steps were crucial in identifying and correcting column name issues in the Synthea CSVs.
3.  **Data Preparation (MTS-Dialog):** Loading a dataset of medical dialogues and corresponding clinical notes. Diagnostic steps were essential to correctly identify the column containing the clinical note text (`section_text`).
4.  **Golden Path Creation:** Selecting a specific sample from the MTS-Dialog dataset as a "Golden Path" for testing the end-to-end process, extracting its dialogue and ground truth note.
5.  **Audio Transcription:** Receiving an audio recording of the "Golden Path" dialogue from the user and transcribing it into text using the Whisper ASR model.

These steps lay the foundation for the next stages of the project, which will likely involve using the generated EHR text files and the transcribed audio to automatically generate clinical notes, potentially using the golden path data for evaluation.

#Day 35

## Install New Dependencies

In [ ]:
!pip install -q torch torchvision torchaudio
!pip install -q pyannote.audio==3.1.1
!pip install -q git+https://github.com/openai/whisper.git
!pip install -q ffmpeg-python soundfile


In [ ]:
from google.colab import userdata

# Fetch your secret token securely
HUGGINGFACE_TOKEN = userdata.get('HF_Token')

if not HUGGINGFACE_TOKEN:
    raise ValueError("Please add your Hugging Face token to Colab secrets with the name 'HUGGINGFACE_TOKEN'.")

# Optional: Log in to Hugging Face Hub
from huggingface_hub import login
login(token=HUGGINGFACE_TOKEN)

In [ ]:
!pip install "numpy<2.0" --force-reinstall
!pip install --upgrade pyannote.audio==3.1.1
!pip install torch torchvision torchaudio --upgrade

In [ ]:
import torchaudio
import torch
from pyannote.audio import Pipeline

# Load audio
waveform, sr = torchaudio.load("/content/Conversation3.mp3")

# Resample to 16kHz if needed
target_sr = 16000
if sr != target_sr:
    waveform = torchaudio.transforms.Resample(sr, target_sr)(waveform)
    sr = target_sr

# Convert to mono
if waveform.shape[0] > 1:
    waveform = torch.mean(waveform, dim=0, keepdim=True)

# Chunk size in samples (e.g., 10 seconds = 160000 samples at 16kHz)
chunk_size = 160000
num_samples = waveform.shape[1]

# Split into chunks
chunks = waveform.unfold(1, chunk_size, chunk_size)  # shape: [1, num_chunks, chunk_size]

In [ ]:
pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-3.1",
    use_auth_token=HUGGINGFACE_TOKEN
)

all_segments = []

for i in range(chunks.shape[1]):
    start_time = i * 10  # seconds
    end_time = start_time + 10

    # Save chunk temporarily (Pyannote works with file paths)
    chunk_path = f"temp_chunk_{i}.wav"
    torchaudio.save(chunk_path, chunks[:, i, :], sr)

    # Run diarization
    diarization_chunk = pipeline(chunk_path)

    # Shift segment times by chunk start time
    for turn, _, speaker in diarization_chunk.itertracks(yield_label=True):
        all_segments.append({
            "start": turn.start + start_time,
            "end": turn.end + start_time,
            "speaker": speaker
        })

In [ ]:
from pyannote.core import Annotation, Segment

annotation = Annotation()
for seg in all_segments:
    annotation[Segment(seg["start"], seg["end"])] = seg["speaker"]

with open("audio.rttm", "w") as rttm_file:
    annotation.write_rttm(rttm_file)

In [ ]:
import whisper
import ffmpeg
from tqdm import tqdm
import os

model = whisper.load_model("small")  # or "base", "tiny" if RAM is limited
AUDIO_PATH = "/content/Conversation3.mp3"

In [ ]:


audio_path = "/content/Conversation3.mp3"
waveform, sr = torchaudio.load(audio_path)

for i, seg in enumerate(all_segments):
    start_sample = int(seg["start"] * sr)
    end_sample = int(seg["end"] * sr)

    speaker_waveform = waveform[:, start_sample:end_sample]

    out_path = f"{seg['speaker']}_segment_{i}.wav"
    torchaudio.save(out_path, speaker_waveform, sr)

In [ ]:
import tempfile
import subprocess

transcript = []

for seg in tqdm(all_segments):
    start = seg["start"]
    end = seg["end"]
    speaker = seg["speaker"]

    # create a temporary file for the audio slice
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=True) as tmpfile:
        (
            ffmpeg
            .input(AUDIO_PATH, ss=start, to=end)
            .output(tmpfile.name, format="wav", ac=1, ar="16k")
            .overwrite_output()
            .run(quiet=True)
        )

        result = model.transcribe(tmpfile.name, fp16=False, language="en")
        text = result["text"].strip()

    transcript.append({
        "speaker": speaker,
        "start": start,
        "end": end,
        "text": text
    })

In [ ]:
speaker_labels = list({seg["speaker"] for seg in transcript})
doctor_speaker = speaker_labels[0]
patient_speaker = speaker_labels[1] if len(speaker_labels) > 1 else speaker_labels[0]

In [ ]:
lines = []
for seg in transcript:
    label = "Doctor" if seg["speaker"] == doctor_speaker else "Patient"
    lines.append(f"[{seg['start']:06.2f}s – {seg['end']:06.2f}s] {label}: {seg['text']}")

output_path = "/content/data/golden_path/conversation_diarized_transcript.txt"

with open(output_path, "w") as f:
    f.write("\n".join(lines))

print(f" Transcript saved to {output_path}")
print("\n".join(lines[:10]))  # preview first few lines

In [ ]:
import json

json_path = "/content/data/golden_path/conversation_diarized.json"
with open(json_path, "w") as f:
    json.dump(transcript, f, indent=2)

print(f"JSON transcript saved to {json_path}")

In [ ]:
import pandas as pd

rttm_path = "audio.rttm"
segments = []

with open(rttm_path, "r") as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) < 9:  # sanity check
            continue
        start = float(parts[3])
        dur = float(parts[4])
        end = start + dur
        speaker = parts[7]
        segments.append({"start": start, "end": end, "speaker": speaker})

df = pd.DataFrame(segments)
df = df.sort_values("start").reset_index(drop=True)
print(df.head())

## Analysis and Summary of Day 35 Execution

Here is a detailed analysis of each executed cell after the "Day 35" markdown heading, explaining what is being done and the outcome of each step, followed by an overall analysis, conclusion, and summary for this section of the notebook:

*   **Install New Dependencies**: This cell installed additional dependencies required for speaker diarization and more advanced audio processing. Key libraries installed were `torch`, `torchvision`, and `torchaudio` (essential for PyTorch-based audio manipulation), `pyannote.audio` (a powerful library for speaker diarization), `openai-whisper` (re-installed potentially to ensure compatibility or get the latest version from the GitHub repository), and `ffmpeg-python` and `soundfile` (for improved audio handling). The output shows successful installation but highlights dependency conflicts related to `numpy` and `pandas` versions. These conflicts suggest potential issues that might arise later if specific versions are strictly required by different libraries.

*   **Fetch Hugging Face Token**: This cell securely fetched a Hugging Face token from Colab's user data secrets. This token is necessary to authenticate and download models from the Hugging Face Hub, specifically the `pyannote/speaker-diarization-3.1` model used later. The `login` function from `huggingface_hub` was used to authenticate the session. The cell includes a check to ensure the token exists, raising an error if not found, which is good practice for handling dependencies.

*   **Resolve Dependency Conflicts**: This cell was added to address the dependency conflicts noted after installing new libraries. It specifically reinstalled `numpy` with a version less than 2.0 (`numpy<2.0`) using `--force-reinstall` to resolve conflicts with `pyannote-metrics` and `pyannote-core`. It also upgraded `pyannote.audio` and `torch`, `torchvision`, `torchaudio` to ensure compatibility after the `numpy` change. The output shows the successful reinstallation of `numpy` to version 1.26.4. However, new dependency conflicts are reported, indicating that forcing a specific `numpy` version might cause conflicts with other pre-installed libraries in the Colab environment (like `jax`, `thinc`, `opencv-python`, etc.). This highlights the challenge of managing dependencies in complex environments.

*   **Load and Chunk Audio**: This cell loaded the audio file (`Conversation3.mp3`) using `torchaudio`. It then checked the sampling rate and resampled the audio to 16kHz if necessary, as many audio processing models, including `pyannote.audio` and Whisper, perform optimally at this rate. It also converted the audio to mono if it was stereo. Finally, it split the audio waveform into chunks of a defined size (10 seconds at 16kHz, i.e., 160000 samples). This chunking strategy is often used for processing long audio files with models that have memory constraints or process audio in segments.

*   **Perform Speaker Diarization (Chunked)**: This cell loaded the `pyannote/speaker-diarization-3.1` pipeline from the Hugging Face Hub using the authenticated token. It then iterated through the audio chunks created in the previous cell. For each chunk, it temporarily saved the chunk as a WAV file (as `pyannote.audio` often works with file paths), performed speaker diarization on that chunk using the loaded pipeline, and then collected the resulting speaker segments. The start and end times of the segments from each chunk were adjusted by adding the start time of the chunk to get the correct timestamps relative to the original full audio file. The output shows that the diarization process started but resulted in an execution failure. The error messages in the stderr output indicate warnings related to deprecated `torchaudio` functions, but the primary cause of the execution failure is not explicitly clear from the provided output. It might be related to resource limitations, model compatibility issues after dependency changes, or an internal error in the diarization pipeline.

*   **Save Diarization Results to RTTM**: This cell processed the collected `all_segments` from the diarization step (despite the previous cell's failure, `all_segments` might contain partial results or the cell might have been run after a successful partial execution or manual intervention). It created a `pyannote.core.Annotation` object from these segments and wrote the diarization results to an RTTM (Rich Transcription Time Mark) file named `audio.rttm`. RTTM is a standard format for storing speaker diarization output, including the start and end times of speech segments and the identified speaker for each segment.

*   **Load Whisper Model**: This cell loaded the Whisper ASR model. It specified the "small" model, noting that a smaller model like "base" or "tiny" could be used if RAM is limited. This model will be used in the next step to transcribe the audio segments.

*   **Slice Audio Segments for Transcription**: This cell attempted to slice the original audio file (`Conversation3.mp3`) into smaller segments based on the speaker diarization results stored in `all_segments`. For each segment identified by the diarization pipeline, it calculated the start and end sample indices based on the original sampling rate (`sr`) and extracted the corresponding waveform slice using `torchaudio`. It then attempted to save each speaker segment as a separate WAV file with a filename indicating the speaker and segment index. The output shows that this cell also resulted in an execution failure, similar to the diarization cell, with warnings about deprecated `torchaudio` functions. The specific cause of the failure here is also not clear from the output but is likely related to issues accessing or processing the audio waveform based on the segments.

*   **Transcribe Diarized Segments using FFmpeg and Whisper**: This cell attempted an alternative approach to slicing and transcribing the audio segments. It iterated through the `all_segments` from the diarization results. For each segment, it used the `ffmpeg-python` library to extract the specific audio slice using the start and end times (`ss` and `to`) and saved it to a temporary WAV file. Then, it used the loaded Whisper model to transcribe the content of this temporary audio slice. The transcription result (`text`) was then stored along with the speaker label and timestamps. This method is often more robust for slicing audio accurately. The output shows a progress bar indicating that the transcription process ran for 5 segments and completed successfully. Despite the failures in the previous two cells, this method of slicing with FFmpeg and transcribing with Whisper seems to have worked for the segments processed.

*   **Identify Speaker Labels**: This cell extracted the unique speaker labels from the `transcript` list generated in the previous transcription step. It then attempted to assign these labels to `doctor_speaker` and `patient_speaker`. It assumes the first identified speaker is the doctor and the second (if present) is the patient. This is a simple heuristic and might not always be accurate, depending on the order in which speakers appear in the diarization output.

*   **Format and Save Diarized Transcript**: This cell formatted the transcribed segments into a human-readable dialogue format. It iterated through the `transcript` list, determined if the speaker was the "Doctor" or "Patient" based on the labels identified in the previous cell, and created a formatted string including the start and end timestamps and the transcribed text for each segment. These formatted lines were then joined together and saved to a text file named `conversation_diarized_transcript.txt` in the `data/golden_path` directory. A preview of the first 10 lines of the saved transcript is printed to the output.

*   **Save Diarized Transcript as JSON**: This cell saved the `transcript` list, which contains the speaker label, start time, end time, and transcribed text for each segment, into a JSON file named `conversation_diarized.json` in the `data/golden_path` directory. Saving the transcript in JSON format provides a structured representation of the data that can be easily parsed and used by other programs or for further processing.

*   **Load and Display RTTM as DataFrame**: This cell loaded the RTTM file (`audio.rttm`) generated in the "Save Diarization Results to RTTM" cell. It parsed the RTTM file line by line to extract the start time, duration, end time, and speaker for each segment. It then created a pandas DataFrame from these extracted segments and sorted the DataFrame by the start time. The head of the resulting DataFrame is printed, showing the structured diarization information. This step confirms that the RTTM file was correctly generated and can be read into a structured format for analysis or further use.

**Overall Analysis, Conclusion, and Summary for Day 35:**

This section of the notebook focused on **speaker diarization and diarized transcription** of the uploaded audio file.

**Analysis:**

*   The initial dependency installation for `pyannote.audio` introduced `numpy` version conflicts, which were attempted to be resolved by forcing a lower `numpy` version. However, this created new conflicts with other existing libraries in the Colab environment, highlighting the complexities of dependency management.
*   Fetching the Hugging Face token was successful, enabling access to the diarization model.
*   The audio was successfully loaded, resampled, converted to mono, and chunked, which is a common preprocessing step for long audio.
*   The `pyannote.audio` speaker diarization pipeline was loaded, but the attempt to run it on the audio chunks resulted in an execution failure. The exact cause of this failure is unclear from the output.
*   Despite the diarization failure on the full chunked audio, the RTTM file was successfully generated and read into a pandas DataFrame. This suggests that either the diarization process completed partially before failing, or the RTTM file was generated from a previous successful run or a different process.
*   The initial attempt to slice audio segments using `torchaudio` and the segments from `all_segments` also failed.
*   A more robust method using `ffmpeg-python` to slice audio segments based on the `all_segments` information and then transcribing each slice using the Whisper model was successful for the segments processed. This indicates that the issue might have been with `torchaudio`'s slicing or handling of the segments, or with the state of `all_segments` after the diarization failure.
*   The transcribed segments were successfully formatted into a human-readable dialogue with speaker labels and timestamps, and also saved in a structured JSON format.

**Conclusion:**

This section successfully demonstrated the process of setting up for speaker diarization and diarized transcription, including dependency management (though with noted challenges), token fetching, audio preprocessing, and using both `pyannote.audio` (partially successful in generating RTTM) and Whisper for transcription. While the direct application of `pyannote.audio` on the chunked audio failed during execution, the use of `ffmpeg-python` for slicing combined with Whisper for transcription provided a working alternative to get the diarized transcript. The RTTM file was also successfully generated and parsed, providing the time-stamped speaker information.

**Summary:**

Day 35 focused on processing the "Golden Path" audio file to obtain a speaker-diarized transcript. Dependencies for audio processing and diarization were installed and configured with a Hugging Face token. The audio was loaded, preprocessed, and chunked. An attempt was made to use `pyannote.audio` for diarization, which generated an RTTM file but failed during execution on the chunked audio. An alternative method using FFmpeg for precise slicing based on the diarization segments and Whisper for transcription was successfully executed, producing a diarized transcript. This transcript was saved in both a human-readable text format and a structured JSON format, along with the RTTM output. This provides a valuable output for the next steps, where this diarized transcript will likely be used in conjunction with the Synthea EHR data and the MTS-Dialog ground truth note for medical note generation and evaluation. The dependency conflicts encountered highlight the need for careful environment management in such projects.